# Feature Engineering

In this section, we construct new predictive features from raw customer data.

The goal is to:
- Capture hidden behavioral patterns
- Improve model interpretability
- Increase predictive power
- Translate business logic into machine learning signals

We design 10 custom features based on:
- Customer tenure behavior
- Service usage patterns
- Billing and payment behavior
- Risk and loyalty indicators

Each feature is accompanied by a short business hypothesis explaining why it may help predict customer churn.

In [3]:
import pandas as pd

df = pd.read_csv("../data/train.csv")

## TenureRange
We convert continuous tenure into meaningful customer lifecycle groups.

In [26]:
bins = [0, 12, 24, 36, 48, 60, float("inf")]
labels = ["0-12", "12-24", "24-36", "36-48", "48-60", "60+"]

df["TenureRange"] = pd.cut(df["tenure"], bins=bins, labels=labels)
df["TenureRange"].value_counts()

TenureRange
0-12     1729
60+      1131
12-24     818
24-36     669
48-60     658
36-48     621
Name: count, dtype: int64

Customer lifecycle grouping (new vs loyal customers).

## TotalServices
We count the number of active services per customer.

In [6]:
service_cols = [
"PhoneService","OnlineSecurity","OnlineBackup",
"DeviceProtection","TechSupport","StreamingTV","StreamingMovies"
]

df["TotalServices"] = (
    (df["PhoneService"] == "Yes").astype(int) +
    (df["OnlineSecurity"] == "Yes").astype(int) +
    (df["OnlineBackup"] == "Yes").astype(int) +
    (df["DeviceProtection"] == "Yes").astype(int) +
    (df["TechSupport"] == "Yes").astype(int) +
    (df["StreamingTV"] == "Yes").astype(int) +
    (df["StreamingMovies"] == "Yes").astype(int)
)

In [22]:
df["TotalServices"].describe()

count    5634.000000
mean        2.958821
std         1.849784
min         0.000000
25%         1.000000
50%         3.000000
75%         4.000000
max         7.000000
Name: TotalServices, dtype: float64

Captures overall service adoption strength.

## Security
Combination of OnlineSecurity and DeviceProtection.

In [23]:
df["Security"] = (
    (df["OnlineSecurity"] == "Yes") &
    (df["DeviceProtection"] == "Yes")
).astype(int)
df["Security"].value_counts()

Security
0    4753
1     881
Name: count, dtype: int64

Detects customers with strong security engagement.

## Entertainment
StreamingTV and StreamingMovies together.

In [24]:
df["Entertainment"] = (
    (df["StreamingTV"] == "Yes") &
    (df["StreamingMovies"] == "Yes")
).astype(int)
df["Entertainment"].value_counts()

Entertainment
0    4057
1    1577
Name: count, dtype: int64

Identifies heavy streaming users.

## SeniorTechSupport
Senior customers with tech support.

In [25]:
df["SeniorTechSupport"] = (
    (df["SeniorCitizen"] == 1) &
    (df["TechSupport"] == "Yes")
).astype(int)
df["SeniorTechSupport"].value_counts()

SeniorTechSupport
0    5424
1     210
Name: count, dtype: int64

Flags vulnerable users needing support.

## BillingAndPayment
High-risk payment behavior pattern.

In [28]:
df["BillingAndPayment"] = (
    (df["PaperlessBilling"] == "Yes") &
    (df["PaymentMethod"] == "Electronic check")
).astype(int)
df["BillingAndPayment"].value_counts()

BillingAndPayment
0    4244
1    1390
Name: count, dtype: int64

High-risk payment behavior indicator.

## MonthlyChargesRange
We discretize MonthlyCharges into fixed-width bins (20 units) to capture pricing segments.

In [27]:
bins = list(range(0, int(df["MonthlyCharges"].max()) + 20, 20))
labels = [f"{i}-{i+20}" for i in bins[:-1]]

df["MonthlyChargesRange"] = pd.cut(df["MonthlyCharges"], bins=bins, labels=labels, include_lowest=True)
df["MonthlyChargesRange"].value_counts()

MonthlyChargesRange
80-100     1423
60-80      1144
20-40       947
40-60       863
100-120     738
0-20        519
Name: count, dtype: int64

Groups customers by pricing tiers.

## AvgMonthlyPerService
We compute cost efficiency per active service.

In [29]:
df["AvgMonthlyPerService"] = df["MonthlyCharges"] / df["TotalServices"]

median_value = df["MonthlyCharges"].median()
df.loc[df["TotalServices"] == 0, "AvgMonthlyPerService"] = median_value

count    5634.000000
mean       26.387755
std        14.361658
min        10.416667
25%        18.383333
50%        20.860000
75%        28.579167
max        77.900000
Name: AvgMonthlyPerService, dtype: float64

## TenureContractRisk
We flag high-risk customers with short tenure and month-to-month contracts.

In [16]:
df["TenureContractRisk"] = (
    (df["Contract"] == "Month-to-month") &
    (df["tenure"] <= 12)
).astype(int)

## LoyaltyFactor
We build a 1–5 loyalty score based on tenure and contract stability.

In [17]:
def loyalty_score(row):
    if row["Contract"] == "Month-to-month":
        if row["tenure"] <= 12:
            return 1
        elif row["tenure"] <= 24:
            return 2
        else:
            return 3
    elif row["Contract"] == "One year":
        if row["tenure"] <= 24:
            return 3
        else:
            return 4
    else:
        return 5

df["LoyaltyFactor"] = df.apply(loyalty_score, axis=1)